# 01 – DRIAMS data exploration (Version 0.1)

Descriptive look at the DRIAMS data that has been downloaded and extracted. **No model is trained here.**

The notebook calls the same functions as `scripts/explore_dataset.py` (in `src/exploration.py`), so the notebook and the script always agree.

Before running: download and extract at least one site (see README), e.g.
```
python scripts/download_driams.py --site B
python scripts/extract_driams.py --site B
```

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import Image, display

from src import exploration as ex
from src.utils import driams_root, load_config, project_path, set_seed

config = load_config(PROJECT_ROOT / "config.yaml")
set_seed(config["project"]["random_seed"])
pd.set_option("display.max_columns", 50)
SPECIES = config["target"]["species"]
PLOTS = project_path(config["paths"]["eda_plots_dir"])
print("DRIAMS root:", driams_root(config))

In [ ]:
sites, missing = ex.load_sites(config)
print("Available:", list(sites), "| not available yet:", missing)
for site, sd in sites.items():
    print(f"{site}: {len(sd.table):,} metadata rows, {len(sd.split.antibiotics)} R/I/S columns, "
          f"0/1 markers {sd.split.binary_markers}, unknown columns {sd.split.unknown}")

## 1. Inventory – metadata rows vs spectrum files

In [ ]:
inventory = ex.inventory_table(sites, config["driams"]["spectra_folders"])
inventory

## 2. Which metadata columns exist at each site?

In [ ]:
presence, categories = ex.column_tables(sites, config["driams"]["metadata_columns"])
display(presence)
categories[categories["category"] != "antibiotic_RIS"]

## 3. Species and duplicates

In [ ]:
d = config["driams"]
species_tbl, species_qc = ex.species_tables(sites, d["failed_identification_species"], d["mixed_species_prefix"])
display(species_qc)
display(species_tbl.groupby("species")[["n_rows", "n_labelled"]].sum().sort_values("n_rows", ascending=False).head(15))
# Leakage checks: duplicates and repeated patients (patient IDs are never shown)
display(ex.duplicate_table(sites, SPECIES, d["group_columns"]).T)
display(ex.group_concentration_table(sites, SPECIES, d["group_columns"]))
ex.acquisition_vs_folder_table(sites)

## 4. Choosing the antibiotic for the target species (pre-registered rules in `config.yaml`)

In [ ]:
candidates, decision = ex.pair_candidates(sites, config)
display(candidates[candidates["pooled_available_class1"] + candidates["pooled_available_class0"] > 0].head(20))
decision

In [ ]:
focus = decision["selected_antibiotic"] or config["target"]["preferred_antibiotic"]
per_year, per_ws = ex.pair_detail_tables(sites, config, focus)
display(per_year)
per_ws if not per_ws.empty else "No workstation column at the available sites."

## 5. Figures
Run `python scripts/explore_dataset.py` first; this cell shows the saved figures.

In [ ]:
figures = sorted(PLOTS.glob("*.png"))
if not figures:
    print("No figures yet - run: python scripts/explore_dataset.py")
for path in figures:
    print(path.name)
    display(Image(filename=str(path), width=900))